In [ ]:
import tensorflow as tf
import numpy as np
import string
import re
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
import os
import requests

In [ ]:
#“Due to limited training data and fewer epochs, the generated text is partially coherent but demonstrates successful learning of character-level language patterns.”

In [ ]:
# data loading
DATA_PATH = "shakespeare.txt"

if not os.path.exists(DATA_PATH):
    print("Downloading dataset...")
    url = "https://www.gutenberg.org/files/100/100-0.txt"
    response = requests.get(url)
    with open(DATA_PATH, "w", encoding="utf-8") as f:
        f.write(response.text)

with open(DATA_PATH, "r", encoding="utf-8") as f:
    text = f.read()
print(f"Original text length: {len(text)}")

Original text length: 5359444


In [ ]:
# 2. Preprocessing
# Convert text to lowercase
text = text.lower()

# Remove punctuation
text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)

# Remove extra whitespaces
text = re.sub(r"\s+", " ", text)

# Use only a subset for faster training
text = text[:200000]

# Create character vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)

char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}

In [ ]:
# 3. Sequence Creation

SEQ_LENGTH = 60

X = []
y = []

for i in range(len(text) - SEQ_LENGTH):
    X.append([char_to_idx[c] for c in text[i:i + SEQ_LENGTH]])
    y.append(char_to_idx[text[i + SEQ_LENGTH]])

X = np.array(X)
y = np.array(y)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=42
)

In [ ]:
# 4. Model

model = Sequential([
    Embedding(vocab_size, 64),
    LSTM(128),
    Dense(vocab_size, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy"
)
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:

# 5. Training

model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=3,
    batch_size=256
)

Epoch 1/3
703/703 ━━━━━━━━━━━━━━━━━━━━ 271s 380ms/step - loss: 2.6132 - val_loss: 2.1060
Epoch 2/3
703/703 ━━━━━━━━━━━━━━━━━━━━ 252s 359ms/step - loss: 2.0524 - val_loss: 1.9372
Epoch 3/3
703/703 ━━━━━━━━━━━━━━━━━━━━ 276s 393ms/step - loss: 1.8952 - val_loss: 1.8337


In [ ]:
# 6. Text Generation
def generate_text(seed, length=300):
    seed = seed.lower()
    generated = seed

    for _ in range(length):
        seq = [char_to_idx.get(c, 0) for c in generated[-SEQ_LENGTH:]]
        seq = tf.keras.preprocessing.sequence.pad_sequences(
            [seq], maxlen=SEQ_LENGTH
        )

        preds = model.predict(seq, verbose=0)[0]
        next_idx = np.random.choice(len(preds), p=preds)
        generated += idx_to_char[next_idx]

    return generated

In [ ]:
# 7. Sample Output
# -------------------------------

print("\n--- Generated Text ---\n")
print(generate_text("to be or not to be "))


--- Generated Text ---

to be or not to be nit in sten ain and you joises grest and my love ford in i wart my in weert he will but mly you in vatemoun in hating not wles exens aplefs in evence’s for allien the coom i sadosy maan bind ungantes of i manter to the that lifewn sa fair thamt on that he comand sit with ow butt to clown if or wow o


In [ ]:
print("\nSeed: the king hath spoken")
print(generate_text("the king hath spoken "))


Seed: the king hath spoken
the king hath spoken will lemt hy loles forst as’s nouk nove whise then veards budt butaged mim as i dayiear eay then glome to wister aroth so frinss of our or dare benaon laster and the pastuon so lond what of has bole and tile in pyour do is bin be from and seed doldsealjed enaf gentare fel to you reare hor agcooms ta
